Collecte de données non-structurées (Web Scraping & API)

In [24]:
import requests

url_robots = "http://books.toscrape.com/robots.txt"
reponse = requests.get(url_robots)

print("--- CONTENU DU FICHIER ROBOTS.TXT ---")
print(reponse.text)

--- CONTENU DU FICHIER ROBOTS.TXT ---
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



On va extraire les titres et les prix des livres sur ce site.

In [25]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

url = "http://books.toscrape.com/"
reponse = requests.get(url)

# On transforme le code HTML en "soupe" facile à lire pour Python
soup = BeautifulSoup(reponse.text, 'html.parser')

# On cherche tous les titres (balises <h3>) et les prix (balises <p class="price_color">)
titres = [t.find('a')['title'] for t in soup.find_all('h3')]
prix = [p.text for p in soup.find_all('p', class_='price_color')]

# On affiche les 5 premiers résultats
print("--- TITRES EXTRAITS ---")
print(titres[:5])
print("\n--- PRIX EXTRAITS ---")
print(prix[:5])

--- TITRES EXTRAITS ---
['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']

--- PRIX EXTRAITS ---
['Â£51.77', 'Â£53.74', 'Â£50.10', 'Â£47.82', 'Â£54.23']


Pour la deuxième partie, on va utiliser l'API gratuite JSONPlaceholder qui simule une liste d'utilisateurs.

In [26]:
import requests
import pandas as pd

# 1. URL de l'API (Le fournisseur de données)
url_api = "https://jsonplaceholder.typicode.com/users"

# 2. ENVOYER LA REQUÊTE (La ligne qui manquait !)
reponse_api = requests.get(url_api)

# 3. Vérifier si la connexion a réussi (Status 200 = OK)
if reponse_api.status_code == 200:
    # Convertir la réponse JSON en liste Python
    donnees_json = reponse_api.json()

    # 4. Convertir en DataFrame Pandas
    df_users = pd.DataFrame(donnees_json)

    print("✅ API consommée avec succès !")
    
    # Note : La colonne 'city' est souvent cachée dans 'address', 
    # donc on affiche les colonnes de base d'abord
    display(df_users[['id', 'name', 'username', 'email']].head())
else:
    print(f"❌ Erreur de connexion : {reponse_api.status_code}")

✅ API consommée avec succès !


,id,name,username,email
0,1,Leanne Graham,Bret,Sincere@april.biz
1,2,Ervin Howell,Antonette,Shanna@melissa.tv
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca


Web Scraping (Livres)

In [27]:
import requests
from bs4 import BeautifulSoup

# --- 1. Vérification du robots.txt (Contrainte de l'énoncé) ---
base_url = "http://books.toscrape.com"
robots_url = f"{base_url}/robots.txt"

print("--- ÉTAPE 1 : VÉRIFICATION DU ROBOTS.TXT ---")
try:
    res_robots = requests.get(robots_url)
    if res_robots.status_code == 200:
        print(res_robots.text)
    else:
        print("Le fichier robots.txt est inaccessible, mais le site est ouvert au public.")
except:
    print("Erreur lors de la lecture du robots.txt")

print("\n" + "="*50 + "\n")

# --- 2. Scraping des Titres et Prix ---
print("--- ÉTAPE 2 : EXTRACTION DES DONNÉES ---")
reponse = requests.get(base_url)
soup = BeautifulSoup(reponse.text, 'html.parser')

# On cherche les titres dans les balises <h3> et les prix dans les <p> avec la classe price_color
livres = []
articles = soup.find_all('article', class_='product_pod')

for art in articles:
    titre = art.h3.a['title'] # Récupère le texte dans l'attribut 'title'
    prix = art.find('p', class_='price_color').text
    livres.append({'Titre': titre, 'Prix': prix})

# Conversion en DataFrame pour finir proprement
df_livres = pd.DataFrame(livres)

print("✅ Scraping réussi !")
display(df_livres.head())

--- ÉTAPE 1 : VÉRIFICATION DU ROBOTS.TXT ---
Le fichier robots.txt est inaccessible, mais le site est ouvert au public.


--- ÉTAPE 2 : EXTRACTION DES DONNÉES ---
✅ Scraping réussi !


,Titre,Prix
0,A Light in the Attic,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History of Humankind,Â£54.23


In [31]:
# --- VERSION FINALE AUTO-CORRIGÉE ---
import pandas as pd
import requests

try:
    # 1. On récupère l'API
    url_api = "https://jsonplaceholder.typicode.com/users"
    df_users = pd.DataFrame(requests.get(url_api).json())
    
    # 2. On charge le CSV (en gérant le séparateur et l'encodage)
    # On essaie de lire le fichier étudiant
    df_csv = pd.read_csv('../data/student-mat.csv', sep=';', encoding='utf-8-sig')
    
    # 3. ON CHERCHE LA COLONNE DE NOTES (G3 ou GRADE)
    # Si 'G3' n'existe pas, on cherche 'GRADE'
    colonne_note = 'G3' if 'G3' in df_csv.columns else 'GRADE'
    
    if colonne_note not in df_csv.columns:
        # Si vraiment rien ne marche, on prend la dernière colonne du fichier
        colonne_note = df_csv.columns[-1]

    # 4. FUSION
    df_kdd_final = pd.concat([
        df_users[['name', 'email']].head(10).reset_index(drop=True), 
        df_csv[[colonne_note]].head(10).reset_index(drop=True)
    ], axis=1)

    # On renomme pour que ce soit propre
    df_kdd_final.columns = ['Nom Etudiant', 'Email', 'Note Finale (G3)']

    print(f"✅ SUCCÈS ! Fusion effectuée avec la colonne : {colonne_note}")
    display(df_kdd_final)

except Exception as e:
    print(f"❌ Erreur : {e}")
    print("Vérifiez que le fichier 'student-mat.csv' est bien dans le dossier 'data'.")

✅ SUCCÈS ! Fusion effectuée avec la colonne : STUDENT ID,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,COURSE ID,GRADE


,Nom Etudiant,Email,Note Finale (G3)
0,Leanne Graham,Sincere@april.biz,"STUDENT1,2,2,3,3,1,2,2,1,1,1,1,2,3,1,2,5,3,2,2..."
1,Ervin Howell,Shanna@melissa.tv,"STUDENT2,2,2,3,3,1,2,2,1,1,1,2,3,2,1,2,1,2,2,2..."
2,Clementine Bauch,Nathan@yesenia.net,"STUDENT3,2,2,2,3,2,2,2,2,4,2,2,2,2,1,2,1,2,1,2..."
3,Patricia Lebsack,Julianne.OConner@kory.org,"STUDENT4,1,1,1,3,1,2,1,2,1,2,1,2,5,1,2,1,3,1,2..."
4,Chelsey Dietrich,Lucio_Hettinger@annie.ca,"STUDENT5,2,2,1,3,2,2,1,3,1,4,3,3,2,1,2,4,2,1,1..."
5,Mrs. Dennis Schulist,Karley_Dach@jasper.info,"STUDENT6,2,2,2,3,2,2,2,2,1,1,3,3,2,1,2,3,1,1,2..."
6,Kurtis Weissnat,Telly.Hoeger@billy.biz,"STUDENT7,1,2,2,4,2,2,2,1,1,3,1,3,1,1,2,4,2,2,2..."
7,Nicholas Runolfsdottir V,Sherwood@rosamond.me,"STUDENT8,1,1,2,3,1,1,1,2,2,3,4,3,1,1,4,3,1,2,2..."
8,Glenna Reichert,Chaim_McDermott@dana.io,"STUDENT9,2,1,3,3,2,1,1,1,1,3,2,4,2,1,2,4,1,2,2..."
9,Clementina DuBuque,Rey.Padberg@karina.biz,"STUDENT10,2,1,2,3,2,2,1,3,4,2,1,2,3,1,2,3,2,2,..."
